In [3]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

# --- Constants ---
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_objects.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32

# --- Tokenizer and PAD_ID ---
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 77
NUM_CATEGORIES = 53
# --- NEW: Number of objects --- <---------------------------------- this number is wrong i think it should be 1000+ need to check this
NUM_OBJECTS = 1013

# --- Granger Causality Matrix Creation ---
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            data = np.vstack([ts_j, ts_i]).T
            try:
                results = grangercausalitytests(data, maxlag=5, verbose=False)
                p_value = results[5][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float)

# --- Dataset and DataLoader ---
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        # The metadata shape is now 4
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# Create Dataset and Loaders
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
g = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

# Execute the matrix creation process
eeg_b, _, _ = next(iter(train_loader))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Creating a static Granger Causality matrix on {device}...")
granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
num_channels = eeg_b.shape[1]

# Add self-loops to handle empty graphs
if granger_edge_index.numel() == 0:
    print("Warning: Generated Granger matrix is empty. Creating a fallback graph with self-loops.")
    granger_edge_index = torch.arange(num_channels, dtype=torch.long).unsqueeze(0).repeat(2, 1)
    granger_edge_attr = torch.ones(num_channels, dtype=torch.float)

granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels)

# Correct the data types before moving to the device
granger_edge_index = granger_edge_index.to(torch.long)
granger_edge_attr = granger_edge_attr.to(torch.float32)

granger_edge_index = granger_edge_index.to(device)
granger_edge_attr = granger_edge_attr.to(device)

print("Granger Matrix created and loaded to device.")

Creating a static Granger Causality matrix on cuda...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Granger Matrix created and loaded to device.


In [9]:
# --- Component 1: SpatioTemporalEEGEncoder (No changes) ---
class SpatioTemporalEEGEncoder(nn.Module):
    # ... (no changes here, same as your code)
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers, bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index += batch_offset.repeat_interleave(edge_index.shape[1]).unsqueeze(0)
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
        temporal_features = x.reshape(batch_size, num_timesteps, -1).permute(1, 0, 2)
        return self.rnn(temporal_features)

# --- Component 2: LuongAttention (No changes) ---
class LuongAttention(nn.Module):
    # ... (no changes here, same as your code)
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)
    def forward(self, decoder_hidden, encoder_outputs):
        scores = torch.bmm(self.attn(encoder_outputs).permute(1, 0, 2), decoder_hidden.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.permute(0, 2, 1), encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(-1)

# --- Component 3: MetadataEncoder ---
# <-- MODIFIED: Replace the entire MetadataEncoder class with this new version

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_categories, num_objects, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128): # Added object_feature_dim
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.category_embedding = nn.Embedding(num_categories, category_emb_dim)
        
        # --- NEW: An MLP to process the raw 1013-element object vector ---
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )
        
        # The output dimension is now the sum of embeddings, the new object feature dim, and the motion score
        self.output_dim = color_emb_dim + category_emb_dim + object_feature_dim + 1

    def forward(self, metadata):
        # metadata shape is (batch_size, 1016)
        color_ids = metadata[:, 0].long()
        category_ids = metadata[:, 1].long()
        motion_values = metadata[:, 2].unsqueeze(1)
        object_features_raw = metadata[:, 3:] # This is the multi-hot vector (batch_size, 1013)

        color_vec = self.color_embedding(color_ids)
        category_vec = self.category_embedding(category_ids)
        
        # --- NEW: Process the raw object vector through the MLP ---
        object_vec = self.object_processor(object_features_raw)
        
        # Concatenate all processed features
        combined_features = torch.cat([color_vec, category_vec, object_vec, motion_values], dim=1)
        return combined_features

# <-- MODIFIED: Replace your entire Decoder class with this one

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        enc_dim = enc_hidden * 2
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)
        # The RNN input now includes the metadata features dimension
        self.rnn = nn.GRU(emb_dim + enc_dim + meta_features_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        # The bridge no longer needs to process metadata, only the encoder hidden state
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden_cat):
        # Only uses the EEG features to initialize the hidden state
        return torch.tanh(self.bridge(encoder_hidden_cat))

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, _ = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        # meta_features are repeated for each token in the sequence (here, just 1)
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        
        # Concatenate metadata with embedding and context at every timestep
        rnn_input = torch.cat((embedded, context.permute(1,0,2), meta_features_unsqueezed), dim=2)
        
        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden

# <-- MODIFIED: Replace your entire Seq2Seq class with this one

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_categories, num_objects, enc_hidden=256, dec_hidden=256, 
                 pad_id=0, dropout=0.2, color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout)
        self.meta_encoder = MetadataEncoder(num_colors, num_categories, num_objects, 
                                            color_emb_dim, category_emb_dim, object_feature_dim)
        
        meta_features_dim = self.meta_encoder.output_dim
        
        self.decoder = Decoder(text_vocab_size, 256, enc_hidden, dec_hidden, 
                               meta_features_dim, 2, pad_id, dropout)
        
        # The input to the meta_head is now just the encoder's output features
        enc_dim = enc_hidden * 2
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256), # Add LayerNorm for stability
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_categories + num_objects + 1)
        )
        self.num_colors = num_colors
        self.num_categories = num_categories
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)
        
        forward_h = encoder_hidden[0::2]
        backward_h = encoder_hidden[1::2]
        encoder_hidden_cat = torch.cat([forward_h, backward_h], dim=2)
        
        # Initialize decoder hidden state using ONLY encoder features
        decoder_hidden = self.decoder.init_hidden(encoder_hidden_cat)
        
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        outputs = torch.zeros(target_len, batch_size, self.decoder.vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]
        
        for t in range(1, target_len):
            # Pass meta_features to the decoder at every timestep
            output, decoder_hidden = self.decoder(decoder_input, decoder_hidden, encoder_outputs, meta_features)
            outputs[t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1
        
        # Predict metadata using ONLY the final encoder hidden state
        meta_preds = self.meta_head(encoder_hidden_cat[-1])
        
        pred_color = meta_preds[:, :self.num_colors]
        pred_category = meta_preds[:, self.num_colors:self.num_colors + self.num_categories]
        pred_object = meta_preds[:, self.num_colors + self.num_categories:self.num_colors + self.num_categories + self.num_objects]
        pred_motion = meta_preds[:, -1]
                
        return outputs[1:].permute(1, 0, 2), pred_color, pred_category, pred_object, pred_motion.squeeze()

In [10]:
# --- Model Instantiation ---
model = Seq2Seq(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS,
    num_categories=NUM_CATEGORIES,
    num_objects=NUM_OBJECTS,
    pad_id=PAD_ID,
    dropout=0.2
).to(device)

print(f"New multi-label model instantiated on '{device}'.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

New multi-label model instantiated on 'cuda'.
Total parameters: 20,014,498


In [12]:
# --- Training Setup ---
text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.CrossEntropyLoss()
category_criterion = nn.CrossEntropyLoss()
# <-- MODIFIED: Switched to BCEWithLogitsLoss for multi-label object classification
object_criterion = nn.BCEWithLogitsLoss() 
motion_criterion = nn.MSELoss()

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)

# <-- MODIFIED: Replace your train_one_epoch and evaluate functions

def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, motion_criterion, granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, meta_b, txt_b = eeg_b.to(device), meta_b.to(device), txt_b.to(device)
        
        optimizer.zero_grad()
        text_logits, pred_color, pred_category, pred_object, pred_motion = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.5)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 3:]) 
        motion_loss = motion_criterion(pred_motion, meta_b[:, 2])
        
        # Adjust loss weights: give object loss a bit more importance
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss + 0.1 * motion_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(text_loss=text_loss.item(), meta_loss=loss.item() - text_loss.item())
        
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, motion_criterion, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    for eeg_b, meta_b, txt_b in loader:
        eeg_b, meta_b, txt_b = eeg_b.to(device), meta_b.to(device), txt_b.to(device)
        text_logits, pred_color, pred_category, pred_object, pred_motion = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 3:])
        motion_loss = motion_criterion(pred_motion, meta_b[:, 2])
        
        # Adjust loss weights
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss + 0.1 * motion_loss
        total_loss += loss.item()
    return total_loss / len(loader)
# --- Training Loop ---
EPOCHS = 20
best_val_loss = float('inf')
print("\n--- Starting Training ---")
for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, motion_criterion, granger_edge_index, granger_edge_attr)
    val_loss = evaluate(model, val_loader, text_criterion, color_criterion, category_criterion, object_criterion, motion_criterion, granger_edge_index, granger_edge_attr)
    scheduler.step(val_loss)
    end_time = time.time()
    formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
    print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time}")
    print(f"\tTrain Loss: {train_loss:.4f}")
    print(f"\t Val. Loss: {val_loss:.4f} | Val. Perplexity: {math.exp(val_loss):7.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'eeg-meta-text-spatiotemporal-corrected-model.pt')
        print("\t-> Validation loss improved, saving new best model.")
print("\n--- Training Complete ---")


--- Starting Training ---


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 01/20 | Time: 06m 58s
	Train Loss: 6.9490
	 Val. Loss: 5.5199 | Val. Perplexity: 249.5990
	-> Validation loss improved, saving new best model.


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 02/20 | Time: 06m 13s
	Train Loss: 5.3442
	 Val. Loss: 5.2050 | Val. Perplexity: 182.1799
	-> Validation loss improved, saving new best model.


...


Epoch 18/20 | Time: 06m 00s
	Train Loss: 3.3113
	 Val. Loss: 4.2109 | Val. Perplexity: 67.4168


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 19/20 | Time: 06m 02s
	Train Loss: 3.2560
	 Val. Loss: 4.1631 | Val. Perplexity: 64.2700
	-> Validation loss improved, saving new best model.


Training:   0%|          | 0/700 [00:00<?, ?it/s]


Epoch 20/20 | Time: 06m 01s
	Train Loss: 3.1992
	 Val. Loss: 4.1297 | Val. Perplexity: 62.1588
	-> Validation loss improved, saving new best model.

--- Training Complete ---


In [15]:
import torch
import torch.nn.functional as F
import random
import json

# --- Load the trained model weights ---
# Make sure this is the model you trained with the corrected code from the last step
checkpoint_path = 'eeg-meta-text-spatiotemporal-corrected-model.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("Best multi-label model loaded successfully.")

# --- Define the CORRECTED multi-label inference function ---
@torch.no_grad()
def generate_multilabel_with_metadata(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=5, max_len=100):
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    meta_signal = meta_signal.unsqueeze(0).to(device)
    
    # --- Feature Encoding ---
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    meta_features = model.meta_encoder(meta_signal)
    
    num_layers = model.decoder.rnn.num_layers
    forward_h = encoder_hidden[0::2]
    backward_h = encoder_hidden[1::2]
    encoder_hidden_cat = torch.cat([forward_h, backward_h], dim=2)
    
    # --- Decoder Initialization & Beam Search ---
    # <-- MODIFIED: Initialize decoder hidden state with ONLY encoder features
    decoder_hidden = model.decoder.init_hidden(encoder_hidden_cat)
    
    beams = [([SOS_ID], 0.0, decoder_hidden)]
    
    for _ in range(max_len):
        new_beams = []
        for seq, score, hidden in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score, hidden))
                continue
            
            input_token = torch.tensor([seq[-1]], device=device)
            # <-- MODIFIED: Pass meta_features into the decoder at every step
            prediction, new_hidden = model.decoder(input_token, hidden, encoder_outputs, meta_features)
            
            log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for i in range(beam_width):
                new_seq = seq + [top_ids[i].item()]
                new_score = score + top_log_probs[i].item()
                new_beams.append((new_seq, new_score, new_hidden))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        if beams[0][0][-1] == EOS_ID:
            break
            
    predicted_text_ids = beams[0][0][1:]
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    
    # --- Metadata Prediction ---
    # <-- MODIFIED: Predict metadata using ONLY the final encoder hidden state
    meta_preds = model.meta_head(encoder_hidden_cat[-1])
    
    pred_color = meta_preds[0, :model.num_colors].argmax().item()
    pred_category = meta_preds[0, model.num_colors:model.num_colors + model.num_categories].argmax().item()
    pred_motion = meta_preds[0, -1].item()
    
    object_logits = meta_preds[0, model.num_colors + model.num_categories:-1]
    object_probs = torch.sigmoid(object_logits)
    pred_objects = (object_probs > 0.5).nonzero(as_tuple=True)[0].tolist()

    return predicted_text, pred_color, pred_category, pred_objects, pred_motion

# --- Run Inference and Print Results ---
NUM_SAMPLES = 10
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name.json"
try:
    with open(OBJECT_MAPPING_FILE, 'r') as f: object_mapping = json.load(f)
    print(f"Object mapping '{OBJECT_MAPPING_FILE}' loaded successfully.")
except FileNotFoundError:
    print(f"Warning: '{OBJECT_MAPPING_FILE}' not found. Object names will not be displayed.")
    object_mapping = {}

print(f"\n--- Running Inference on the First {NUM_SAMPLES} Samples ---")

for i in range(NUM_SAMPLES):
    eeg_sample, meta_sample, true_text_ids = test_ds[i]
    
    true_color_id = int(meta_sample[0].item())
    true_category_id = int(meta_sample[1].item())
    true_motion = meta_sample[2].item()
    true_object_ids = meta_sample[3:].nonzero(as_tuple=True)[0].tolist()

    predicted_text, pred_color, pred_category, pred_object_ids, pred_motion = generate_multilabel_with_metadata(
        model, eeg_sample, meta_sample, granger_edge_index, granger_edge_attr
    )
    
    true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)

    true_object_names = [object_mapping.get(str(oid), f"ID:{oid}") for oid in true_object_ids]
    pred_object_names = [object_mapping.get(str(oid), f"ID:{oid}") for oid in pred_object_ids]

    print(f"\n--- Sample {i+1}/{NUM_SAMPLES} (Index: {i}) ---")
    print(f"GROUND TRUTH TEXT: {true_text}")
    print(f"MODEL PREDICTION TEXT: {predicted_text}")
    print("\nMETADATA PREDICTION:")
    print(f"  Color ID:      Truth={true_color_id}, Predicted={pred_color}")
    print(f"  Category ID:   Truth={true_category_id}, Predicted={pred_category}")
    print(f"  Objects:       Truth={true_object_names}, Predicted={pred_object_names}")
    print(f"  Motion:        Truth={true_motion:.4f}, Predicted={pred_motion:.4f}")

Best multi-label model loaded successfully.
Object mapping '/home/poorna/data/object_id_to_name.json' loaded successfully.

--- Running Inference on the First 10 Samples ---

--- Sample 1/10 (Index: 0) ---
GROUND TRUTH TEXT: a school of orange fish swims around a vibrant coral reef.. tone : serene
MODEL PREDICTION TEXT: a sea shark swims gracefully in a vibrant coral reef.. tone : serene

METADATA PREDICTION:
  Color ID:      Truth=67, Predicted=36
  Category ID:   Truth=36, Predicted=50
  Objects:       Truth=['coral reef', 'fish', 'ocean'], Predicted=[]
  Motion:        Truth=0.4000, Predicted=0.2375

--- Sample 2/10 (Index: 1) ---
GROUND TRUTH TEXT: powerful waterfalls cascade down rocky cliffs into a misty pool.. tone : awe - inspiring
MODEL PREDICTION TEXT: a car drives down a sandy beach at sunset.. tone : serene

METADATA PREDICTION:
  Color ID:      Truth=36, Predicted=36
  Category ID:   Truth=50, Predicted=50
  Objects:       Truth=['cliff', 'mist', 'rocks', 'trees', 'water',